# Financial Statement Audit Results

> Includes Big 4 Vietnam verification from extracted audit firm names.

This notebook reads audit extraction outputs from `financial_statement_audit_results` in DuckDB (read-only).

It includes:
- DB connection
- table existence and quick stats
- filtered result query
- Big 4 Vietnam verification
- one-record detail view
- optional CSV export

In [1]:
from pathlib import Path
import json

import duckdb
import pandas as pd

try:
    from config import DB_PATH as APP_DB_PATH
    DB_PATH = Path(APP_DB_PATH)
    db_source = 'config.DB_PATH'
except Exception:
    DB_PATH = Path('/media/nvme0n1/dev/annual_report/db.db')
    db_source = 'fallback literal path'

assert DB_PATH.exists(), f'Database file not found: {DB_PATH}'
con = duckdb.connect(str(DB_PATH), read_only=True)
con.execute('PRAGMA threads=4;')

print('Connected to:', DB_PATH)
print('DB source:', db_source)
print('Context:', con.execute("SELECT current_database(), current_schema()").fetchall())

Connected to: /media/nvme0n1/dev/annual_report/db.db
DB source: config.DB_PATH
Context: [('db', 'main')]


In [2]:
required_tables = [
    'financial_statement_audit_results',
    'financial_statement_audit_jobs',
    'financial_statement_reports',
]

tables_df = con.execute(
    """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY table_name
    """
).df()
available_tables = set(tables_df['table_name'].tolist())
missing_tables = [t for t in required_tables if t not in available_tables]

print('Total tables:', len(available_tables))
print('Missing required tables:', missing_tables)
display(tables_df[tables_df['table_name'].isin(required_tables)])

stats = con.execute(
    """
    SELECT
        COUNT(*) AS total_results,
        COUNT(DISTINCT ticker) AS total_tickers,
        MIN(year) AS min_year,
        MAX(year) AS max_year,
        SUM(CASE WHEN found THEN 1 ELSE 0 END) AS found_count
    FROM financial_statement_audit_results
    """
).df()
display(stats)

models_df = con.execute(
    """
    SELECT model, COUNT(*) AS rows
    FROM financial_statement_audit_results
    GROUP BY model
    ORDER BY rows DESC, model
    """
).df()
display(models_df)

Total tables: 37
Missing required tables: []


,table_name
9,financial_statement_audit_jobs
10,financial_statement_audit_results
12,financial_statement_reports


,total_results,total_tickers,min_year,max_year,found_count
0,687,65,2015,2025,685.0


,model,rows
0,gpt-4.1-mini,686
1,gpt-5-mini,1


In [9]:
# Filters: set to None to disable that filter
ticker_filter = None      # Example: 'VNM'
year_from = None          # Example: 2020
year_to = None            # Example: 2025
model_filter = "gpt-4.1-mini"       # Example: 'gpt-5-mini'
created_after = None      # Example: '2026-08-01'
limit_rows = 50000

where_clauses = ['1=1']
params = []

if ticker_filter:
    where_clauses.append('ticker = ?')
    params.append(str(ticker_filter).upper())

if year_from is not None:
    where_clauses.append('year >= ?')
    params.append(int(year_from))

if year_to is not None:
    where_clauses.append('year <= ?')
    params.append(int(year_to))

if model_filter:
    where_clauses.append('model = ?')
    params.append(str(model_filter))

if created_after:
    where_clauses.append('created_at >= ?')
    params.append(str(created_after))

params.append(int(limit_rows))

sql = f"""
SELECT
    ticker,
    year,
    audit_firm,
    signing_auditor_names
FROM financial_statement_audit_results
WHERE {' AND '.join(where_clauses)} AND created_at > '2026-08-01'
ORDER BY ticker, year DESC, created_at DESC
LIMIT ?
"""

results_df = con.execute(sql, params).df()

def _parse_json_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    text = str(value).strip()
    if not text:
        return []
    try:
        loaded = json.loads(text)
        return loaded if isinstance(loaded, list) else [loaded]
    except Exception:
        return [text]

def _normalize_vi_text(value):
    import unicodedata
    text = str(value or '').strip().lower()
    text = unicodedata.normalize('NFD', text)
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    return text.replace('đ', 'd')

def _big4_vn_label(audit_firm):
    text = _normalize_vi_text(audit_firm)
    if not text:
        return None

    # Big 4 matching with common legal-name variants in Vietnam
    if ('deloitte' in text) or ('dttl' in text):
        return 'Deloitte'
    if ('ernst' in text and 'young' in text) or (' ey ' in f' {text} ') or text.startswith('ey '):
        return 'EY'
    if ('pricewaterhousecoopers' in text) or ('pwc' in text):
        return 'PwC'
    if 'kpmg' in text:
        return 'KPMG'
    return None

if not results_df.empty:
    results_df['signing_auditor_names_list'] = results_df['signing_auditor_names'].apply(_parse_json_list)
    results_df['signing_auditor_count'] = results_df['signing_auditor_names_list'].apply(len)
    results_df['big4_firm_group'] = results_df['audit_firm'].apply(_big4_vn_label)
    results_df['is_big4_vn'] = results_df['big4_firm_group'].notna().astype(int)

print('Rows returned:', len(results_df))
display(results_df.head(100))

if not results_df.empty:
    summary_df = (
        results_df.groupby('is_big4_vn', dropna=False)
        .size()
        .reset_index(name='rows')
        .sort_values('is_big4_vn', ascending=False)
    )
    firm_breakdown_df = (
        results_df.assign(big4_firm_group=results_df['big4_firm_group'].fillna('Non-Big4/Unknown'))
        .groupby('big4_firm_group', dropna=False)
        .size()
        .reset_index(name='rows')
        .sort_values('rows', ascending=False)
    )
    print('Big 4 verification summary:')
    display(summary_df)
    print('Firm group breakdown:')
    display(firm_breakdown_df)

Rows returned: 685


,ticker,year,audit_firm,signing_auditor_names,signing_auditor_names_list,signing_auditor_count,big4_firm_group,is_big4_vn
0,AAA,2025,Công ty Trách nhiệm Hữu hạn Ernst & Young Việt...,"[""Nguyễn Hoàng Linh"", ""Ngô Thị Phương Nhung""]","[Nguyễn Hoàng Linh, Ngô Thị Phương Nhung]",2,EY,1
1,AAA,2024,Công ty Trách nhiệm Hữu hạn Ernst & Young Việt...,[],[],0,EY,1
2,AAA,2023,Công ty Trách nhiệm Hữu hạn Ernst & Young Việt...,[],[],0,EY,1
3,AAA,2022,Công ty Trách nhiệm Hữu hạn Ernst & Young Việt...,[],[],0,EY,1
4,AAA,2021,Công ty Trách nhiệm Hữu hạn Ernst & Young Việt...,"[""Phùng Manh Phú"", ""Lê Tuấn Trung""]","[Phùng Manh Phú, Lê Tuấn Trung]",2,EY,1
...,...,...,...,...,...,...,...,...
95,CIA,2022,Công ty TNHH Hãng Kiểm toán AASC,"[""Đỗ Mạnh Cường"", ""Đỗ Thị Hồng Thủy""]","[Đỗ Mạnh Cường, Đỗ Thị Hồng Thủy]",2,NaN,0
96,CIA,2021,Công ty TNHH Hãng Kiểm toán AASC,"[""Đỗ Thị Ngọc Dung"", ""Đinh Quang Trung""]","[Đỗ Thị Ngọc Dung, Đinh Quang Trung]",2,NaN,0
97,CIA,2019,Công ty TNHH Hãng Kiểm toán AASC,"[""Đinh Quang Trung""]",[Đinh Quang Trung],1,NaN,0
98,CIA,2018,Công ty TNHH Hãng Kiểm toán AASC,"[""Đỗ Mạnh Cường"", ""Đỗ Thị Hồng Thủy""]","[Đỗ Mạnh Cường, Đỗ Thị Hồng Thủy]",2,NaN,0


Big 4 verification summary:


,is_big4_vn,rows
1,1,212
0,0,473


Firm group breakdown:


,big4_firm_group,rows
3,Non-Big4/Unknown,473
1,EY,76
0,Deloitte,53
2,KPMG,48
4,PwC,35


In [12]:
results_df[['ticker', 'year', 'audit_firm', 'big4_firm_group', 'is_big4_vn']]

,ticker,year,audit_firm,big4_firm_group,is_big4_vn
0,AAA,2025,Công ty Trách nhiệm Hữu hạn Ernst & Young Việt...,EY,1
1,AAA,2024,Công ty Trách nhiệm Hữu hạn Ernst & Young Việt...,EY,1
2,AAA,2023,Công ty Trách nhiệm Hữu hạn Ernst & Young Việt...,EY,1
3,AAA,2022,Công ty Trách nhiệm Hữu hạn Ernst & Young Việt...,EY,1
4,AAA,2021,Công ty Trách nhiệm Hữu hạn Ernst & Young Việt...,EY,1
...,...,...,...,...,...
680,WCS,2019,Công ty TNHH Kiểm toán AFC Việt Nam,NaN,0
681,WCS,2018,Công ty TNHH Kiểm toán AFC Việt Nam,NaN,0
682,WCS,2017,Công ty TNHH Kiểm toán AFC Việt Nam,NaN,0
683,WCS,2016,Công ty TNHH Kiểm Toán AFC Việt Nam,NaN,0


In [11]:
duckdb.execute("""SELECT distinct ticker FROM df WHERE ticker in ('GMD',
'GSP',
'HCT',
'HTV',
'HVN',
'MAC',
'MAS',
'MHC',
'NAP',
'PGT',
'PJC',
'PJT',
'PRC',
'PTS',
'PVP',
'SKG',
'SRF',
'TCO',
'TJC',
'TOT',
'VIP',
'VJC',
'VNS',
'VNT',
'VOS',
'VTO') ORDER BY ticker
""").df()

,ticker
0,GMD
1,GSP
2,HCT
3,HTV
4,HVN
5,MAC
6,MAS
7,MHC
8,NAP
9,PGT


In [5]:
# Detail view for one ticker/year (latest row if multiple models exist)
if 'results_df' in globals() and not results_df.empty:
    detail_ticker = str(results_df.iloc[0]['ticker'])
    detail_year = int(results_df.iloc[0]['year'])
else:
    detail_ticker = 'AAA'
    detail_year = 2015

print('Using detail filter:', detail_ticker, detail_year)

detail_sql = """
SELECT
    ticker,
    year,
    found,
    audit_firm,
    audit_opinion,
    signing_auditor_names,
    value_json,
    details_json,
    reason,
    top_chunks,
    similarities,
    model,
    created_at
FROM financial_statement_audit_results
WHERE ticker = ? AND year = ?
ORDER BY created_at DESC
"""

detail_df = con.execute(detail_sql, [detail_ticker.upper(), int(detail_year)]).df()
print('Detail rows:', len(detail_df))
display(detail_df)

# Optional export
export_csv = False
if export_csv:
    out_path = Path('data/output/financial_statement_audit_results.csv')
    out_path.parent.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(out_path, index=False)
    print('Saved:', out_path.resolve())

Using detail filter: AAA 2015
Detail rows: 1


,ticker,year,found,audit_firm,audit_opinion,signing_auditor_names,value_json,details_json,reason,top_chunks,similarities,model,created_at
0,AAA,2015,True,Công ty TNHH Kiểm toán và Tư vấn Tài chính Quố...,unqualified,"[""Nguyễn Như Phương"", ""Khúc Đình Dũng"", ""Đoàn ...","{""external_audit_firm"": ""Công ty TNHH Kiểm toá...","[{""name"": ""Công ty TNHH Kiểm toán và Tư vấn Tà...",Independent audit report titled 'Báo cáo kiểm ...,"[9, 11, 5, 8, 7, 6, 0, 24, 103, 129, 20, 23, 1...","[0.613531, 0.647959, 0.663406, 0.568923, 0.619...",gpt-5-mini,2026-08-11 22:04:49.575142
